# UHPC Flexural Strength — Probabilistic Decision Support System
## GP_ARD_RBF Prediction with Uncertainty Quantification & Reliability Assessment

**Pipeline:**
- Feature engineering (17 → 26 features, 9 physics-informed)
- Boruta feature selection (training set only)
- 5-fold cross-validation (calibration + accuracy metrics)
- Final GP_ARD_RBF trained on full training set
- Interactive dashboard: prediction · gauge · β-index table · reliability curve

In [1]:
# !pip install boruta ipywidgets scikit-learn pandas numpy matplotlib scipy

In [2]:
import numpy as np
import pandas as pd
import warnings
from scipy import stats
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from boruta import BorutaPy

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Circle, Wedge
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, clear_output, HTML

import ipywidgets as widgets
from ipywidgets import (
    FloatText, Button, HBox, VBox,
    Output, Layout, Label, HTML as HTMLWidget
)

warnings.filterwarnings('ignore')
print('All packages imported successfully!')

All packages imported successfully!


In [3]:
# ============================================================
#
# ============================================================
COLORS = {
    'primary'   : '#1B6CA8',   # Nature blue
    'secondary' : '#C0392B',   # academic red
    'teal'      : '#138D75',   # forest teal
    'amber'     : '#D4A017',   # warm amber
    'slate'     : '#5D6D7E',   # neutral slate
    'lavender'  : '#7D7FBF',   # soft purple
    'bg'        : '#F7F9FC',   # off-white background
    'text'      : '#1C2833',   # near-black text
    'grid'      : '#D5D8DC',   # light grid
    'pass_green': '#1E8449',   # pass indicator
    'fail_red'  : '#C0392B',   # fail indicator
}

plt.rcParams.update({
    'font.family'        : 'DejaVu Sans',
    'font.size'          : 11,
    'axes.titlesize'     : 13,
    'axes.titleweight'   : 'bold',
    'axes.labelsize'     : 11,
    'axes.labelweight'   : 'bold',
    'axes.linewidth'     : 1.2,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'xtick.labelsize'    : 9,
    'ytick.labelsize'    : 9,
    'legend.fontsize'    : 9,
    'figure.facecolor'   : 'white',
    'axes.facecolor'     : '#F7F9FC',
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
})

# Global model state
gp_model        = None
scaler          = None
selected_feats  = None
all_feat_names  = None
model_metrics   = {}

print('Colour palette and settings configured.')

Colour palette and settings configured.


In [4]:
# ============================================================
# Utility: calibration metrics
# ============================================================
def picp(y_true, mu, sigma, alpha=0.95):
    z = norm.ppf(0.5 + alpha / 2)
    return float(np.mean((y_true >= mu - z*sigma) & (y_true <= mu + z*sigma)))

def ece_metric(y_true, mu, sigma, M=19):
    alphas = np.linspace(0.50, 0.99, M)
    return float(np.mean([abs(picp(y_true, mu, sigma, a) - a) for a in alphas]))

def exceedance_prob(mean, std, threshold):
    """P(FS > threshold) using GP posterior"""
    return float((1 - norm.cdf((threshold - mean) / (std + 1e-9))) * 100)

def reliability_beta(mean, std, threshold):
    """Reliability index beta = Phi^{-1}(P(FS > threshold))"""
    p = exceedance_prob(mean, std, threshold) / 100
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return float(norm.ppf(p))

print('Utility functions defined.')

Utility functions defined.


In [5]:
# ============================================================
# Feature engineering 
# ============================================================
RAW_FEATURES = ['C','SF','QP','FA','SL','MK','S','QS',
                 'W','SP','Age','L','D','BV','PPV','GV','SSV']
TARGET = 'FS'
EPS = 1e-6

def engineer_features(df):
    """Add 9 physics-informed features to dataframe (in-place copy)."""
    d = df.copy()
    # --- 6 general (same as CS paper) ---
    d['W_C_ratio']       = d['W'] / (d['C'] + EPS)
    d['Total_Binder']    = d['C'] + d['SF'] + d['FA'] + d['SL'] + d['MK']
    d['W_Binder_ratio']  = d['W'] / (d['Total_Binder'] + EPS)
    d['SF_C_ratio']      = d['SF'] / (d['C'] + EPS)
    d['Total_Fiber']     = d['BV'] + d['PPV'] + d['GV'] + d['SSV']
    d['Fiber_Aspect']    = d['L'] / (d['D'] + EPS)
    # --- 2 flexural-specific ---
    d['Fiber_RI']        = d['Total_Fiber'] * d['Fiber_Aspect']          # reinforcement index
    d['Steel_Efficiency']= d['SSV'] * d['L'] / (d['D'] + EPS)           # steel fibre efficiency
    # --- 1 effective W/B with pozzolan correction ---
    d['W_Beff']          = d['W'] / (d['C'] + 0.5*d['SF'] + 0.3*d['FA']
                                      + 0.4*d['SL'] + 0.6*d['MK'] + EPS)
    return d

ENG_FEATURES = ['W_C_ratio','Total_Binder','W_Binder_ratio','SF_C_ratio',
                 'Total_Fiber','Fiber_Aspect','Fiber_RI','Steel_Efficiency','W_Beff']
ALL_FEATURES = RAW_FEATURES + ENG_FEATURES  # 26 total

print(f'Feature sets defined: {len(RAW_FEATURES)} raw + {len(ENG_FEATURES)} engineered = {len(ALL_FEATURES)} total.')

Feature sets defined: 17 raw + 9 engineered = 26 total.


In [6]:
# ============================================================
# Model training: Boruta + 5-fold CV + final GP_ARD_RBF
# ============================================================
def build_ard_rbf(n_features):
    return (ConstantKernel(1.0, (1e-3, 1e4)) *
            RBF(length_scale=np.ones(n_features),
                length_scale_bounds=(1e-2, 1e4)) +
            WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e2)))


def train_gp_model(data_path='uhpc_flexural_strength_1.csv',
                   n_cv_folds=5, seed=42, n_restarts=5):
    global gp_model, scaler, selected_feats, all_feat_names, model_metrics

    print('=' * 68)
    print('  UHPC Flexural Strength — Probabilistic Decision Support System')
    print('  GP_ARD_RBF · Boruta Selection · 5-Fold CV')
    print('=' * 68)

    # ── Step 1: load & engineer ───────────────────────────────────────────
    print('\n[1/5] Loading data and engineering features...')
    df_raw = pd.read_csv(data_path)
    print(f'  Dataset: {len(df_raw)} samples × {df_raw.shape[1]} columns')

    df_eng = engineer_features(df_raw)
    X_all  = df_eng[ALL_FEATURES].values
    y_all  = df_eng[TARGET].values
    all_feat_names = ALL_FEATURES
    print(f'  Features: {len(RAW_FEATURES)} raw + {len(ENG_FEATURES)} engineered = {len(ALL_FEATURES)} total')
    print(f'  Target FS: {y_all.min():.2f}–{y_all.max():.2f} MPa  (mean {y_all.mean():.2f} MPa)')

    # ── Step 2: 80/20 split ───────────────────────────────────────────────
    X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
        X_all, y_all, test_size=0.20, random_state=seed)
    print(f'  Split: {len(y_tr)} train / {len(y_te)} test  (80/20, seed={seed})')

    # ── Step 3: Boruta on training set ────────────────────────────────────
    print('\n[2/5] Boruta feature selection on training set...')
    scaler_boruta = StandardScaler()
    X_tr_sc_all   = scaler_boruta.fit_transform(X_tr_raw)

    rf_b = RandomForestRegressor(n_estimators=500, random_state=seed, n_jobs=-1)
    sel  = BorutaPy(rf_b, n_estimators='auto', alpha=0.05,
                    max_iter=100, random_state=seed, verbose=0)
    sel.fit(X_tr_sc_all, y_tr)

    mask = sel.support_ | sel.support_weak_   # confirmed + tentative
    selected_feats = [ALL_FEATURES[i] for i in range(len(ALL_FEATURES)) if mask[i]]
    sel_idx        = [ALL_FEATURES.index(f) for f in selected_feats]
    n_sel          = len(selected_feats)

    confirmed  = [f for f in selected_feats if sel.support_[ALL_FEATURES.index(f)]]
    tentative  = [f for f in selected_feats if sel.support_weak_[ALL_FEATURES.index(f)]]
    rejected   = [ALL_FEATURES[i] for i in range(len(ALL_FEATURES)) if not mask[i]]
    print(f'  Confirmed ({len(confirmed)}): {confirmed}')
    if tentative:
        print(f'  Tentative ({len(tentative)}): {tentative}')
    print(f'  Rejected  ({len(rejected)}): {rejected}')
    print(f'  → {n_sel}/{len(ALL_FEATURES)} features retained')

    # Final scaler on selected features only
    scaler     = StandardScaler()
    X_tr_sel   = scaler.fit_transform(X_tr_raw[:, sel_idx])
    X_te_sel   = scaler.transform(X_te_raw[:, sel_idx])

    # ── Step 4: 5-fold CV on training set ─────────────────────────────────
    print(f'\n[3/5] {n_cv_folds}-fold cross-validation  '
          f'(GP_ARD_RBF, {n_sel} features, {n_restarts} restarts/fold)...')
    kf = KFold(n_splits=n_cv_folds, shuffle=True, random_state=seed)
    cv_r2, cv_rmse, cv_mae, cv_picp, cv_ece = [], [], [], [], []

    for fold, (tr_idx, vl_idx) in enumerate(kf.split(X_tr_sel), 1):
        Xf_tr, Xf_vl = X_tr_sel[tr_idx], X_tr_sel[vl_idx]
        yf_tr, yf_vl = y_tr[tr_idx],     y_tr[vl_idx]

        gp_f = GaussianProcessRegressor(
            kernel=build_ard_rbf(n_sel), alpha=0.0,
            n_restarts_optimizer=n_restarts,
            normalize_y=True, random_state=seed)
        gp_f.fit(Xf_tr, yf_tr)
        mu_vl, sig_vl = gp_f.predict(Xf_vl, return_std=True)

        r2   = r2_score(yf_vl, mu_vl)
        rmse = float(np.sqrt(mean_squared_error(yf_vl, mu_vl)))
        mae  = float(mean_absolute_error(yf_vl, mu_vl))
        p95  = picp(yf_vl, mu_vl, sig_vl)
        ece  = ece_metric(yf_vl, mu_vl, sig_vl)

        cv_r2.append(r2);  cv_rmse.append(rmse);  cv_mae.append(mae)
        cv_picp.append(p95);  cv_ece.append(ece)
        print(f'  Fold {fold}  R\u00b2={r2:.4f}  RMSE={rmse:.3f} MPa  '
              f'PICP\u2085={p95*100:.1f}%  ECE={ece:.4f}')

    print(f'  ── CV Summary ────────────────────────────────────────────────')
    print(f'  R\u00b2    = {np.mean(cv_r2):.4f} \u00b1 {np.std(cv_r2):.4f}')
    print(f'  RMSE  = {np.mean(cv_rmse):.3f} \u00b1 {np.std(cv_rmse):.3f} MPa')
    print(f'  PICP\u2085 = {np.mean(cv_picp)*100:.1f}% \u00b1 {np.std(cv_picp)*100:.1f}%')
    print(f'  ECE   = {np.mean(cv_ece):.4f} \u00b1 {np.std(cv_ece):.4f}')

    # ── Step 5: final GP on full training set ─────────────────────────────
    print(f'\n[4/5] Training final GP_ARD_RBF on {len(y_tr)} samples...')
    gp_model = GaussianProcessRegressor(
        kernel=build_ard_rbf(n_sel), alpha=0.0,
        n_restarts_optimizer=n_restarts,
        normalize_y=True, random_state=seed)
    gp_model.fit(X_tr_sel, y_tr)
    print(f'  Kernel: {gp_model.kernel_}')

    print(f'\n[5/5] Test set evaluation ({len(y_te)} samples)...')
    mu_te, sig_te = gp_model.predict(X_te_sel, return_std=True)
    te_r2   = r2_score(y_te, mu_te)
    te_rmse = float(np.sqrt(mean_squared_error(y_te, mu_te)))
    te_mae  = float(mean_absolute_error(y_te, mu_te))
    te_p95  = picp(y_te, mu_te, sig_te)
    te_ece  = ece_metric(y_te, mu_te, sig_te)
    te_mpiw = float(np.mean(2 * norm.ppf(0.975) * sig_te))

    model_metrics = dict(
        cv_r2=f'{np.mean(cv_r2):.3f}\u00b1{np.std(cv_r2):.3f}',
        cv_picp=f'{np.mean(cv_picp)*100:.1f}%\u00b1{np.std(cv_picp)*100:.1f}%',
        cv_ece=f'{np.mean(cv_ece):.3f}\u00b1{np.std(cv_ece):.3f}',
        te_r2=te_r2, te_rmse=te_rmse, te_mae=te_mae,
        te_p95=te_p95, te_ece=te_ece, te_mpiw=te_mpiw,
        n_features=n_sel,
    )

    print(f'  R\u00b2={te_r2:.4f}  RMSE={te_rmse:.3f} MPa  '
          f'PICP\u2085={te_p95*100:.1f}%  ECE={te_ece:.4f}  MPIW={te_mpiw:.2f} MPa')
    print('\n' + '=' * 68)
    print('  Model ready. Launch the dashboard below.')
    print('=' * 68)
    return True

In [7]:
# ============================================================
# Prediction function (auto-computes engineered features)
# ============================================================
def predict_fs(inputs: dict):
    """Return (mean_MPa, std_MPa) from raw 17-feature input dict."""
    global gp_model, scaler, selected_feats, all_feat_names
    if gp_model is None:
        raise RuntimeError('Model not trained. Run train_gp_model() first.')

    # Build single-row dataframe of raw features
    row = {f: inputs.get(f, 0.0) for f in RAW_FEATURES}
    df_row = pd.DataFrame([row])
    df_row = engineer_features(df_row)

    # Align to ALL_FEATURES order
    x_full = df_row[ALL_FEATURES].values  # shape (1, 26)

    # Select Boruta columns, scale, predict
    sel_idx = [ALL_FEATURES.index(f) for f in selected_feats]
    x_sel   = scaler.transform(x_full[:, sel_idx])
    mu, sig = gp_model.predict(x_sel, return_std=True)
    return float(mu[0]), float(sig[0])

In [8]:
# ============================================================
# Visualization helpers
# ============================================================

def _gauge_color(value):
    if value >= 80: return COLORS['teal']
    if value >= 50: return COLORS['primary']
    if value >= 20: return COLORS['amber']
    return COLORS['secondary']


def plot_gauge(ax, value, label):
    """Semi-circular probability gauge."""
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-0.4, 1.6)
    ax.set_aspect('equal')
    ax.axis('off')

    ax.add_patch(Wedge((0,0), 1.0, 0, 180, width=0.28,
                        facecolor=COLORS['grid'], edgecolor='none'))
    col = _gauge_color(value)
    ax.add_patch(Wedge((0,0), 1.0, 0, value/100*180, width=0.28,
                        facecolor=col, edgecolor='none', alpha=0.90))
    ax.add_patch(Circle((0,0), 0.68, facecolor='white',
                          edgecolor=COLORS['grid'], linewidth=1.8))

    ax.text(0,  0.18, f'{value:.1f}%', ha='center', va='center',
            fontsize=26, fontweight='bold', color=col)
    ax.text(0, -0.12, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color=COLORS['text'])
    for xp, txt in [(-1.25, '0'), (0, '50'), (1.25, '100')]:
        ax.text(xp, 0.02, txt, ha='center', fontsize=8.5, color='#888')


def plot_prediction_box(ax, mean, std):
    """Prediction result card."""
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    ax.axis('off')

    ax.text(5, 9.3, 'GP Prediction Result', ha='center',
            fontsize=14, fontweight='bold', color=COLORS['primary'])

    ax.add_patch(FancyBboxPatch((0.8, 3.8), 8.4, 4.5,
                                 boxstyle='round,pad=0.15,rounding_size=0.6',
                                 facecolor='white', edgecolor=COLORS['primary'],
                                 linewidth=2.2))

    ax.text(5, 6.8, f'{mean:.2f}', ha='center', va='center',
            fontsize=38, fontweight='bold', color=COLORS['primary'])
    ax.text(5, 5.5, f'\u00b1 {std:.2f} MPa', ha='center', va='center',
            fontsize=17, fontweight='bold', color=COLORS['text'])
    ax.text(5, 4.6, '(posterior std)', ha='center', fontsize=10, color='#777')

    lo, hi = mean - 1.96*std, mean + 1.96*std
    ax.text(5, 2.8, f'95% CI:  [{lo:.1f},  {hi:.1f}] MPa',
            ha='center', fontsize=12, fontweight='bold', color=COLORS['teal'])
    ax.text(5, 1.6, 'Predicted Flexural Strength',
            ha='center', fontsize=10, color='#888')


def plot_beta_table(ax, mean, std, thresholds=(10, 15, 20, 25, 30, 35)):
    """Exceedance probability + reliability index beta table."""
    ax.axis('off')

    rows = []
    for d in thresholds:
        p   = exceedance_prob(mean, std, d)
        b   = reliability_beta(mean, std, d)
        rows.append([f'{d} MPa', f'{p:.1f}%', f'{b:.3f}'])

    col_labels = ['Threshold', 'P(FS > d)', '\u03b2 index']
    tbl = ax.table(cellText=rows, colLabels=col_labels,
                   cellLoc='center', loc='center',
                   colWidths=[0.34, 0.33, 0.33])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10.5)
    tbl.scale(1.1, 2.0)

    # Header
    for j in range(3):
        tbl[(0, j)].set_facecolor(COLORS['primary'])
        tbl[(0, j)].set_text_props(color='white', fontweight='bold')

    # Row colouring by probability
    for i, d in enumerate(thresholds):
        p = exceedance_prob(mean, std, d)
        color = ('#E8F5E9' if p >= 80 else
                 '#E3F2FD' if p >= 50 else
                 '#FFF8E1' if p >= 20 else '#FFEBEE')
        for j in range(3):
            tbl[(i+1, j)].set_facecolor(color)

    ax.set_title('Exceedance Probability & \u03b2 Index\n'
                 '(fib MC 2020 target: \u03b2 = 3.8)',
                 fontsize=11, fontweight='bold', color=COLORS['text'], pad=8)


def plot_reliability_curve(ax, mean, std, threshold=20.0):
    """Full-width reliability curve with current threshold marker."""
    lo_x = max(2.0,  mean - 4.5*std)
    hi_x = min(65.0, mean + 4.5*std)
    xs   = np.linspace(lo_x, hi_x, 300)
    ys   = [exceedance_prob(mean, std, x) for x in xs]

    ax.fill_between(xs, ys, alpha=0.18, color=COLORS['primary'])
    ax.plot(xs, ys, color=COLORS['primary'], lw=2.4, label='Reliability curve')

    for ref_y, col, ls, lbl in [(95, COLORS['secondary'], '--', '95% target'),
                                  (50, COLORS['slate'],    ':',  '50% reference')]:
        ax.axhline(ref_y, color=col, ls=ls, lw=1.4, alpha=0.8, label=lbl)

    # Current threshold marker
    cur_p = exceedance_prob(mean, std, threshold)
    ax.axvline(threshold, color=COLORS['secondary'], lw=1.8, alpha=0.85)
    ax.scatter([threshold], [cur_p], s=120, color=COLORS['secondary'],
               zorder=6, edgecolor='white', lw=1.8)
    ax.annotate(f'P = {cur_p:.1f}%\n@ {threshold:.0f} MPa',
                xy=(threshold, cur_p),
                xytext=(threshold + max(1.5, (hi_x-lo_x)*0.08), cur_p + 8),
                fontsize=10.5, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                          edgecolor=COLORS['secondary'], alpha=0.95),
                arrowprops=dict(arrowstyle='->', color=COLORS['secondary'], lw=1.4))

    # Mean line
    ax.axvline(mean, color=COLORS['primary'], ls=':', lw=1.8, alpha=0.75)
    ax.text(mean, 3, f'\u03bc={mean:.1f}', ha='center', fontsize=9.5,
            fontweight='bold', color=COLORS['primary'])

    ax.set_xlabel('Design Strength Threshold (MPa)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Exceedance Probability  P(FS > d)  [%]', fontsize=11, fontweight='bold')
    ax.set_title('Reliability Curve for Design Decision Support',
                 fontsize=13, fontweight='bold', color=COLORS['primary'])
    ax.set_xlim(lo_x, hi_x)
    ax.set_ylim(0, 100)
    ax.legend(loc='upper right', fontsize=9.5, framealpha=0.9)
    ax.grid(True, alpha=0.25, ls='--')

In [9]:
# ============================================================
# Interactive Dashboard
# ============================================================
class UHPCFlexuralDashboard:

    # Default and example mix proportions
    DEFAULTS = dict(C=900, SF=150, QP=100, FA=0, SL=0, MK=0,
                    S=1000, QS=0, W=180, SP=25, Age=28,
                    L=13.0, D=0.20, BV=0.0, PPV=0.0, GV=0.0, SSV=2.0)
    EXAMPLE  = dict(C=950, SF=180, QP=120, FA=0, SL=0, MK=0,
                    S=980, QS=0, W=170, SP=28, Age=28,
                    L=13.0, D=0.20, BV=0.0, PPV=0.0, GV=0.0, SSV=2.0)

    def __init__(self):
        self.out = Output()
        self._build_widgets()
        self._build_layout()

    # ── widget creation ──────────────────────────────────────────────────
    def _build_widgets(self):
        ilay = Layout(width='110px')
        self.inp = {k: widgets.FloatText(value=v, layout=ilay)
                    for k, v in self.DEFAULTS.items()}

        self.slider = widgets.FloatSlider(
            value=20, min=5, max=50, step=1,
            description='', continuous_update=False,
            layout=Layout(width='360px'))

        btn_kw = dict(layout=Layout(width='130px', height='42px'))
        self.btn_predict = Button(description='PREDICT',
                                  button_style='', **btn_kw)
        self.btn_predict.style.button_color = COLORS['primary']
        self.btn_predict.style.font_weight  = 'bold'

        self.btn_clear   = Button(description='CLEAR',
                                  button_style='warning', **btn_kw)
        self.btn_example = Button(description='EXAMPLE',
                                  button_style='info', **btn_kw)

        self.btn_predict.on_click(self._on_predict)
        self.btn_clear.on_click(self._on_clear)
        self.btn_example.on_click(self._on_example)

        self.status = HTMLWidget(
            value='<span style="color:#1B6CA8;font-weight:bold;">Ready</span>')

    # ── layout assembly ──────────────────────────────────────────────────
    def _build_layout(self):

        def row(label, key, unit):
            return HBox([Label(f'{label} ({unit}):',
                               layout=Layout(width='155px')), self.inp[key]])

        header = HTMLWidget(value='''
        <div style="background:linear-gradient(135deg,#1B6CA8 0%,#138D75 100%);
                    padding:18px 24px;border-radius:10px;margin-bottom:12px;">
          <h1 style="color:white;margin:0;font-size:22px;font-weight:700;">
            UHPC Flexural Strength — Probabilistic Decision Support</h1>
          <p style="color:#D6EAF8;margin:4px 0 0;font-size:13px;">
            GP_ARD_RBF · Boruta Selection · 5-Fold CV · \u03b2 Reliability Index</p>
        </div>''')

        col1 = VBox([
            HTMLWidget(value='<b style="color:#1B6CA8;">Cementitious</b>'),
            row('Cement',       'C',  'kg/m\u00b3'),
            row('Silica Fume',  'SF', 'kg/m\u00b3'),
            row('Quartz Powder','QP', 'kg/m\u00b3'),
            row('Fly Ash',      'FA', 'kg/m\u00b3'),
            row('Slag',         'SL', 'kg/m\u00b3'),
            row('Metakaolin',   'MK', 'kg/m\u00b3'),
        ])
        col2 = VBox([
            HTMLWidget(value='<b style="color:#138D75;">Aggregates & Water</b>'),
            row('Sand',          'S',   'kg/m\u00b3'),
            row('Quartz Sand',   'QS',  'kg/m\u00b3'),
            row('Water',         'W',   'kg/m\u00b3'),
            row('Superplast.',   'SP',  'kg/m\u00b3'),
            row('Curing Age',    'Age', 'days'),
        ])
        col3 = VBox([
            HTMLWidget(value='<b style="color:#C0392B;">Fibre Parameters</b>'),
            row('Fibre Length',  'L',   'mm'),
            row('Fibre Diameter','D',   'mm'),
            row('Basalt Fibre',  'BV',  'vol%'),
            row('Polypropylene', 'PPV', 'vol%'),
            row('Glass Fibre',   'GV',  'vol%'),
            row('Steel Fibre',   'SSV', 'vol%'),
        ])

        threshold_block = VBox([
            HTMLWidget(value='''
            <div style="background:#EBF5FB;padding:8px 12px;
                        border-left:4px solid #1B6CA8;border-radius:4px;
                        margin:12px 0 6px;">
              <b style="color:#1B6CA8;font-size:13px;">
                Design Strength Threshold (MPa)</b></div>'''),
            HBox([Label('Threshold:', layout=Layout(width='80px')),
                  self.slider,
                  Label('MPa')])
        ])

        btns = HBox(
            [self.btn_predict, self.btn_clear, self.btn_example, self.status],
            layout=Layout(justify_content='center', margin='12px 0'))

        # Model performance badge (populated after training)
        self.perf_badge = HTMLWidget(value=self._perf_html())

        self.ui = VBox([
            header,
            self.perf_badge,
            HTMLWidget(value='''
            <div style="background:#F2F3F4;padding:8px 12px;border-radius:5px;">
              <b style="color:#1C2833;font-size:13px;">Mix Design Parameters</b>
            </div>'''),
            HBox([col1, col2, col3],
                 layout=Layout(justify_content='space-around')),
            threshold_block,
            btns,
            HTMLWidget(value='''
            <div style="background:#EAF2FF;padding:8px 12px;border-radius:5px;">
              <b style="color:#1B6CA8;font-size:13px;">Prediction Dashboard</b>
            </div>'''),
            self.out,
        ])

    # ── performance badge ────────────────────────────────────────────────
    def _perf_html(self):
        if not model_metrics:
            return ''
        m = model_metrics
        return (f'<div style="background:#F0FFF4;border:1px solid #A9DFBF;'
                f'padding:8px 16px;border-radius:6px;font-size:12px;">'
                f'<b style="color:#1E8449;">Model validated</b> — '
                f'5-fold CV: R\u00b2={m["cv_r2"]}  PICP\u2085={m["cv_picp"]}  '
                f'ECE={m["cv_ece"]} &nbsp;|&nbsp; '
                f'Test: R\u00b2={m["te_r2"]:.3f}  RMSE={m["te_rmse"]:.2f} MPa  '
                f'PICP\u2085={m["te_p95"]*100:.1f}%  ECE={m["te_ece"]:.3f}  '
                f'({m["n_features"]} features)'
                f'</div>')

    # ── button callbacks ─────────────────────────────────────────────────
    def _on_predict(self, _):
        with self.out:
            clear_output(wait=True)
            try:
                inputs    = {k: w.value for k, w in self.inp.items()}
                threshold = float(self.slider.value)
                mean, std = predict_fs(inputs)

                # Derived quantities
                steel_eff = inputs['SSV'] * inputs['L'] / (inputs['D'] + EPS)

                # ── figure layout: 2 rows × 3 cols ──────────────────────
                fig = plt.figure(figsize=(15, 10), facecolor='white')
                gs  = gridspec.GridSpec(2, 3,
                                        height_ratios=[1, 1.3],
                                        hspace=0.42, wspace=0.32)

                ax_pred  = fig.add_subplot(gs[0, 0])
                ax_gauge = fig.add_subplot(gs[0, 1])
                ax_table = fig.add_subplot(gs[0, 2])
                ax_curve = fig.add_subplot(gs[1, :])

                plot_prediction_box(ax_pred, mean, std)
                plot_gauge(ax_gauge, exceedance_prob(mean, std, threshold),
                           f'P(FS > {threshold:.0f} MPa)')
                plot_beta_table(ax_table, mean, std)
                plot_reliability_curve(ax_curve, mean, std, threshold)

                # Steel efficiency annotation in panel (a)
                ax_pred.text(5, 0.5,
                             f'Steel Efficiency = {steel_eff:.1f}  '
                             f'(SSV\u00d7L/D)',
                             ha='center', fontsize=9, color='#888',
                             style='italic')

                fig.suptitle(
                    'UHPC Flexural Strength — Probabilistic Analysis Dashboard',
                    fontsize=15, fontweight='bold',
                    color=COLORS['primary'], y=1.01)
                plt.tight_layout(rect=[0, 0, 1, 0.98])
                plt.show()

                # Console summary
                beta_thr = reliability_beta(mean, std, threshold)
                prob_thr = exceedance_prob(mean, std, threshold)
                print('\n' + '─' * 65)
                print(f'  Predicted FS  : {mean:.2f} \u00b1 {std:.2f} MPa')
                print(f'  95% CI        : [{mean-1.96*std:.1f},  {mean+1.96*std:.1f}] MPa')
                print(f'  P(FS > {threshold:.0f} MPa) : {prob_thr:.1f}%')
                print(f'  \u03b2 index       : {beta_thr:.3f}  '
                      f'(fib MC 2020 target = 3.8)')
                print(f'  Steel Efficiency: {steel_eff:.1f}  (SSV\u00d7L/D)')
                print('─' * 65)

                self.status.value = (
                    '<span style="color:#138D75;font-weight:bold;">Done \u2713</span>')
                self.perf_badge.value = self._perf_html()

            except Exception as e:
                print(f'Error: {e}')
                self.status.value = (
                    '<span style="color:#C0392B;font-weight:bold;">Error</span>')

    def _on_clear(self, _):
        for k, v in self.DEFAULTS.items():
            self.inp[k].value = v
        self.slider.value = 20
        with self.out:
            clear_output()
        self.status.value = (
            '<span style="color:#1B6CA8;font-weight:bold;">Cleared</span>')

    def _on_example(self, _):
        for k, v in self.EXAMPLE.items():
            self.inp[k].value = v
        self.slider.value = 25
        with self.out:
            clear_output()
            print('Example mix loaded — click PREDICT.')
        self.status.value = (
            '<span style="color:#D4A017;font-weight:bold;">Example loaded</span>')

    def show(self):
        display(self.ui)

---
## Step 1 — Train the GP model
**Ensure `uhpc_flexural_strength_1.csv` is in the same directory as this notebook.**  
Training takes approximately 3–6 minutes (5-fold CV × 5 optimisation restarts per fold).

In [10]:
train_gp_model('uhpc_flexural_strength_1.csv')

  UHPC Flexural Strength — Probabilistic Decision Support System
  GP_ARD_RBF · Boruta Selection · 5-Fold CV

[1/5] Loading data and engineering features...
  Dataset: 321 samples × 18 columns
  Features: 17 raw + 9 engineered = 26 total
  Target FS: 7.78–55.51 MPa  (mean 27.50 MPa)
  Split: 256 train / 65 test  (80/20, seed=42)

[2/5] Boruta feature selection on training set...
  Confirmed (11): ['SF', 'S', 'QS', 'W', 'SP', 'Age', 'D', 'SSV', 'SF_C_ratio', 'Fiber_RI', 'Steel_Efficiency']
  Tentative (3): ['SL', 'W_C_ratio', 'W_Beff']
  Rejected  (12): ['C', 'QP', 'FA', 'MK', 'L', 'BV', 'PPV', 'GV', 'Total_Binder', 'W_Binder_ratio', 'Total_Fiber', 'Fiber_Aspect']
  → 14/26 features retained

[3/5] 5-fold cross-validation  (GP_ARD_RBF, 14 features, 5 restarts/fold)...
  Fold 1  R²=0.7385  RMSE=4.277 MPa  PICP₅=92.3%  ECE=0.0200
  Fold 2  R²=0.7634  RMSE=5.131 MPa  PICP₅=96.1%  ECE=0.0267
  Fold 3  R²=0.7193  RMSE=5.330 MPa  PICP₅=84.3%  ECE=0.0434
  Fold 4  R²=0.7080  RMSE=6.162 MPa  PI

True

---
## Step 2 — Launch the interactive dashboard

In [11]:
dashboard = UHPCFlexuralDashboard()
dashboard.show()

---
## Quick prediction (optional — no GUI)
Direct function call for scripted use or batch testing.

In [12]:
sample = dict(C=950, SF=180, QP=120, FA=0, SL=0, MK=0,
              S=980, QS=0, W=170, SP=28, Age=28,
              L=13.0, D=0.20, BV=0.0, PPV=0.0, GV=0.0, SSV=2.0)

mu, sigma = predict_fs(sample)
print(f'Predicted FS  : {mu:.2f} \u00b1 {sigma:.2f} MPa')
print(f'95% CI        : [{mu-1.96*sigma:.1f},  {mu+1.96*sigma:.1f}] MPa')
print()
for d in [10, 15, 20, 25, 30, 35]:
    p = exceedance_prob(mu, sigma, d)
    b = reliability_beta(mu, sigma, d)
    print(f'  P(FS > {d:2d} MPa) = {p:5.1f}%   beta = {b:+.3f}')

Predicted FS  : 37.80 ± 10.89 MPa
95% CI        : [16.4,  59.1] MPa

  P(FS > 10 MPa) =  99.5%   beta = +2.552
  P(FS > 15 MPa) =  98.2%   beta = +2.093
  P(FS > 20 MPa) =  94.9%   beta = +1.634
  P(FS > 25 MPa) =  88.0%   beta = +1.175
  P(FS > 30 MPa) =  76.3%   beta = +0.716
  P(FS > 35 MPa) =  60.1%   beta = +0.257
